# Flight Delay Project — Feature Engineering

In this notebook, we create new variables that make the flight data easier to analyze.

We will:
- create time-based features from flight date
- build a delay flag and delay severity categories
- create route-level information
- combine delay cause columns into a useful summary feature
- inspect the data before saving the final featured dataset

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path().resolve().parent
data_path = project_root / "data" / "processed" / "flights_cleaned.csv"

df = pd.read_csv(data_path)

df.head()

,FL_DATE,AIRLINE,ORIGIN,DEST,DEP_DELAY,ARR_DELAY,CANCELLED,DIVERTED,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,FLL,EWR,-4.0,-14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022-11-19,Delta Air Lines Inc.,MSP,SEA,-6.0,-5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2022-07-22,United Air Lines Inc.,DEN,MSP,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2023-03-06,Delta Air Lines Inc.,MSP,SFO,-1.0,24.0,0.0,0.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,MCO,DFW,-2.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Convert flight date to datetime format
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

In [10]:
# Create time-based features
df['month'] = df['FL_DATE'].dt.month
df['day_of_week'] = df['FL_DATE'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

In [11]:
# Create a delay flag:
# 1 = delayed more than 15 minutes
# 0 = on time or minor delay
df['delay_flag'] = (df['ARR_DELAY'] > 15).astype(int)

In [12]:
# Create a delay severity label
def delay_category(x):
    if x <= 15:
        return "On Time"
    elif x <= 60:
        return "Moderate Delay"
    else:
        return "Severe Delay"

df['delay_category'] = df['ARR_DELAY'].apply(delay_category)

In [6]:
df['route'] = df['ORIGIN'] + "-" + df['DEST']

In [13]:
# Create a total delay components feature
delay_cols = [
    'DELAY_DUE_CARRIER',
    'DELAY_DUE_WEATHER',
    'DELAY_DUE_NAS',
    'DELAY_DUE_SECURITY',
    'DELAY_DUE_LATE_AIRCRAFT'
]

df['total_delay_components'] = df[delay_cols].sum(axis=1)

In [14]:
# Create a quick summary column for analysis
df["delay_minus_dep"] = df["ARR_DELAY"] - df["DEP_DELAY"]

print("Feature engineering complete.")
print("Current shape:", df.shape)

Feature engineering complete.
Current shape: (2913802, 21)


In [15]:
# Show a quick preview of the new features
df.head()

# Check new columns and their types
df.info()

# Check whether the new features look sensible
df[[
    "FL_DATE",
    "AIRLINE",
    "ORIGIN",
    "DEST",
    "ARR_DELAY",
    "delay_flag",
    "delay_category",
    "route",
    "month",
    "day_of_week",
    "is_weekend",
    "total_delay_components"
]].head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2913802 entries, 0 to 2913801
Data columns (total 21 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   FL_DATE                  datetime64[ns]
 1   AIRLINE                  object        
 2   ORIGIN                   object        
 3   DEST                     object        
 4   DEP_DELAY                float64       
 5   ARR_DELAY                float64       
 6   CANCELLED                float64       
 7   DIVERTED                 float64       
 8   DELAY_DUE_CARRIER        float64       
 9   DELAY_DUE_WEATHER        float64       
 10  DELAY_DUE_NAS            float64       
 11  DELAY_DUE_SECURITY       float64       
 12  DELAY_DUE_LATE_AIRCRAFT  float64       
 13  month                    int32         
 14  day_of_week              int32         
 15  is_weekend               int64         
 16  delay_flag               int64         
 17  delay_category           ob

,FL_DATE,AIRLINE,ORIGIN,DEST,ARR_DELAY,delay_flag,delay_category,route,month,day_of_week,is_weekend,total_delay_components
0,2019-01-09,United Air Lines Inc.,FLL,EWR,-14.0,0,On Time,FLL-EWR,1,2,0,0.0
1,2022-11-19,Delta Air Lines Inc.,MSP,SEA,-5.0,0,On Time,MSP-SEA,11,5,1,0.0
2,2022-07-22,United Air Lines Inc.,DEN,MSP,0.0,0,On Time,DEN-MSP,7,4,0,0.0
3,2023-03-06,Delta Air Lines Inc.,MSP,SFO,24.0,1,Moderate Delay,MSP-SFO,3,0,0,24.0
4,2020-02-23,Spirit Air Lines,MCO,DFW,-1.0,0,On Time,MCO-DFW,2,6,1,0.0
5,2019-07-31,Southwest Airlines Co.,DAL,OKC,141.0,1,Severe Delay,DAL-OKC,7,2,0,141.0
6,2023-06-11,American Airlines Inc.,DCA,BOS,-29.0,0,On Time,DCA-BOS,6,6,1,0.0
7,2019-07-08,Republic Airline,HSV,DCA,23.0,1,Moderate Delay,HSV-DCA,7,0,0,23.0
8,2023-02-12,Spirit Air Lines,IAH,LAX,-11.0,0,On Time,IAH-LAX,2,6,1,0.0
9,2020-08-22,Alaska Airlines Inc.,SEA,FAI,1.0,0,On Time,SEA-FAI,8,5,1,0.0


In [16]:
# Basic checks on the new columns
print("Delay category counts:")
print(df["delay_category"].value_counts())

print("\nWeekend split:")
print(df["is_weekend"].value_counts())

print("\nDelay flag split:")
print(df["delay_flag"].value_counts())

Delay category counts:
delay_category
On Time           2398513
Moderate Delay     340154
Severe Delay       175135
Name: count, dtype: int64

Weekend split:
is_weekend
0    2116271
1     797531
Name: count, dtype: int64

Delay flag split:
delay_flag
0    2398513
1     515289
Name: count, dtype: int64


In [8]:
output_path = project_root / "data" / "processed" / "flights_featured.csv"
df.to_csv(output_path, index=False)